In [ ]:
pip install tensorflow opencv-python numpy matplotlib scikit-learn pydicom Pillow 

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt
import json
import os 

# --- Configuración (Asegúrate que coincida con el entrenamiento) ---
MODEL_PATH = './renal_disease_classifier_mobilenetv2.h5' # Ruta al modelo guardado
LABELS_PATH = './renal_disease_classifier_mobilenetv2_labels.json' # Ruta a las etiquetas guardadas
IMG_WIDTH, IMG_HEIGHT = 224, 224 # Mismo tamaño que en el entrenamiento

# --- Cargar Modelo y Etiquetas ---
print(f"Cargando modelo desde {MODEL_PATH}...")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Archivo de modelo no encontrado en: {MODEL_PATH}")
loaded_model = load_model(MODEL_PATH)
print("Modelo cargado.")

print(f"Cargando etiquetas desde {LABELS_PATH}...")
if not os.path.exists(LABELS_PATH):
     raise FileNotFoundError(f"Archivo de etiquetas no encontrado en: {LABELS_PATH}")
with open(LABELS_PATH, 'r') as f:
    # Convertir las claves (que son strings en JSON) de nuevo a enteros
    class_labels_map = {int(k): v for k, v in json.load(f).items()}
print("Etiquetas cargadas:", class_labels_map)

# --- Función de Preprocesamiento ---
def preprocess_single_image(img_path, target_w, target_h):
    """Carga y preprocesa una sola imagen para el modelo."""
    try:
        img = image.load_img(img_path, target_size=(target_w, target_h))
        img_array = image.img_to_array(img)
        # Normalizar (¡Asegúrate que coincida con el entrenamiento!)
        img_array = img_array / 255.0
        # Añadir dimensión de batch
        img_batch = np.expand_dims(img_array, axis=0)
        return img_batch, img # Devuelve también la imagen original para mostrarla
    except Exception as e:
        print(f"Error al cargar o preprocesar la imagen {img_path}: {e}")
        return None, None

# --- Cargar y Predecir una Imagen de Prueba ---
# Reemplaza con la ruta a tu imagen
IMAGE_TO_TEST = '/Dataset/proob/image.png' # O .jpg, etc.

if not os.path.exists(IMAGE_TO_TEST):
     print(f"ADVERTENCIA: La imagen de prueba no se encontró en {IMAGE_TO_TEST}. Por favor, actualiza la ruta.")
else:
    processed_image, original_image = preprocess_single_image(IMAGE_TO_TEST, IMG_WIDTH, IMG_HEIGHT)

    if processed_image is not None:
        # Realizar la predicción
        print("Realizando predicción...")
        predictions = loaded_model.predict(processed_image)
        # predictions = [[prob_clase0, prob_clase1, prob_clase2, prob_clase3]]

        # Obtener la clase predicha y la confianza
        predicted_index = np.argmax(predictions[0])
        confidence = np.max(predictions[0]) * 100
        predicted_label = class_labels_map.get(predicted_index, "Clase Desconocida") # Usa el mapeo cargado

        print(f"\nPredicción: {predicted_label}")
        print(f"Confianza: {confidence:.2f}%")
        # print("Probabilidades por clase:", predictions[0]) # Descomenta para ver todas las probabilidades

        # Mostrar la imagen original con la predicción
        plt.figure(figsize=(6,6))
        plt.imshow(original_image) # Muestra la imagen cargada por load_img
        plt.title(f"Predicción: {predicted_label} ({confidence:.2f}%)")
        plt.axis('off')
        plt.show()
